# Data Science for Business - Stacking on Micro Mortgages

*Session 7 · Not covered in ISLP; builds on ISLP 5.1.3 (k-fold CV), 8.2.1 (averaging reduces variance), 6.2.2 (the lasso), 8.2.2 (random forests) and 2.2.3 (KNN)*

Stacking goes beyond ISLP. For further reading, see Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, Section 8.8 "Model Averaging and Stacking".

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

In [ ]:
np.random.seed(42)

# scikit-learn >= 1.8 announces that `penalty='l1'` will be replaced by `l1_ratio=1`. We keep penalty='l1',
# because it is the spelling that also gives a lasso on older versions (e.g., on Colab), and silence only these two notices.
warnings.filterwarnings('ignore', message="'penalty' was deprecated", category=FutureWarning)
warnings.filterwarnings('ignore', message="Inconsistent values: penalty=", category=UserWarning)

## Problem description

In India, there are about 20 million home loan (mortgage) aspirants
working in the informal sector:

- Monthly income between INR 20,000-25,000 (\$ 325-400)
- Typically no formal accounts and documents (e.g., tax returns, income proofs, bank statements)
- Often use services of money lenders with interest rates between 30 and 60% per annum

Providing mortgages to this group of customers requires to quickly and
efficiently assess their creditworthiness. Due to a lack of formal
documents and objective data, most financial institutions perform
interview-based processes to decide about these loan requests:

Strength of the current process:

-   Interview-based field assessment

-   Relaxation of document requirements

Weaknesses of the current process:

-   Costly (total transaction costs as high as 30% of loan volume)

-   Subjective judgments; depends on individual skills and motivations

-   Low reliability across branches and credit officers

-   Risk of corruption and fraud

The data comes from a Harvard Business Review case. Our goal: predict the credit officers' decision (`Decision`: 1 = approved, 0 = rejected) from the application data. See `variables.pdf` for a description of the variables (source: Rudravaram et al., 2013).

## Load data

Load training data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_07/micromortgage.csv')

## Prepare data

We use the same preparation steps as in the tree-based models notebook: drop the ID, make `Tier` categorical, and do a stratified train/test split (80% of the requests are approved).

In [ ]:
data = data.drop(['ID'], axis=1)
data["Tier"] = data["Tier"].apply(lambda x: "T"+str(x))

All monetary variables (loan amount, down payment, market value of the property, bank savings, and the monthly income, expenses and installments) are in Indian rupees, with values of up to several million. We express them in units of **100,000 rupees** (one *lakh*, a common unit in India). This makes the coefficients of our models much easier to read (e.g., the effect of 100,000 rupees more down payment instead of one rupee more) and avoids numerical problems when fitting models; it does not change the predictions. The mortgage ratios (`IIR`, `IAR`, `FOIR`, `LTV`, `LVR`) are percentages and stay as they are.

In [ ]:
money_columns = ['LoanReq', 'Dwnpay', 'MarVal', 'BankSave', 'TotInc', 'Expen', 'DispInc', 'CalcEmi', 'OldEmi']
data[money_columns] = data[money_columns] / 100000   # in 100,000 rupees

In [ ]:
X = data.drop("Decision", axis=1)
y = data["Decision"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Preprocessing pipeline

The categorical predictors are one-hot encoded and the numerical predictors standardized. The lasso needs standardized predictors, because its penalty depends on the scale of the coefficients (ISLP 6.2), and so does KNN, because it measures distances between observations. For the random forest, scaling is unnecessary but harmless, so all three base models can share the same preprocessing. As in the tree-based models notebook, we combine the preprocessing with each model in a `Pipeline`.

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

preprocessor = ColumnTransformer(
                transformers=[
                        ('num', StandardScaler(), numerical_features),
                        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
                    ], verbose_feature_names_out=False
                ).set_output(transform='pandas')

## Base models

Stacking combines **different learning algorithms**. We use three very different base models, each tuned with 5-fold cross-validation and the AUC as the criterion:

- a **lasso** logistic regression (ISLP 6.2.2): `penalty='l1'` makes `LogisticRegression` a lasso; `C` is the inverse of λ,
- the **random forest** from the tree-based models notebook (ISLP 8.2.2),
- a **K-nearest neighbors** classifier (ISLP 2.2.3).

In [ ]:
model_lasso_tuned = GridSearchCV(
                estimator=Pipeline([
                        ('preprocess', preprocessor),
                        ('model', LogisticRegression(penalty='l1', solver='liblinear', max_iter=10000, random_state=42))
                    ]),
                param_grid={
                        'model__C': np.logspace(-4, 4, 10)
                    }, scoring='roc_auc', cv=5, n_jobs=-1
                )

In [ ]:
model_lasso_tuned.fit(X_train, y_train)
pred_lasso_proba = model_lasso_tuned.predict_proba(X_test)[:,1]
roc_auc_score(y_test, pred_lasso_proba)

In [ ]:
model_rf_tuned = GridSearchCV(
                estimator=Pipeline([
                        ('preprocess', preprocessor),
                        ('model', RandomForestClassifier(n_estimators=500, random_state=42))
                    ]),
                param_grid={
                        'model__max_features': ['sqrt', 0.33, None]
                    }, scoring='roc_auc', cv=5, n_jobs=-1
                )

In [ ]:
model_rf_tuned.fit(X_train, y_train)
pred_rf_proba = model_rf_tuned.predict_proba(X_test)[:,1]
roc_auc_score(y_test, pred_rf_proba)

In [ ]:
model_knn_tuned = GridSearchCV(
                estimator=Pipeline([
                        ('preprocess', preprocessor),
                        ('model', KNeighborsClassifier())
                    ]),
                param_grid={
                        'model__n_neighbors': [5, 15, 31, 61]
                    }, scoring='roc_auc', cv=5, n_jobs=-1
                )

In [ ]:
model_knn_tuned.fit(X_train, y_train)
pred_knn_proba = model_knn_tuned.predict_proba(X_test)[:,1]
roc_auc_score(y_test, pred_knn_proba)

## Combine models by averaging

The simplest way to combine models is to average their predicted probabilities with equal weights (*blending*). Like bagging, averaging reduces variance (ISLP 8.2.1), and it helps most when the models make different errors. But every model gets the same say, including the weakest one (here: KNN).

In [ ]:
pred_avg_proba = (pred_lasso_proba + pred_rf_proba + pred_knn_proba) / 3
roc_auc_score(y_test, pred_avg_proba)

## Stacking with a meta-learner

In **stacking**, a second-level model, the **meta-learner**, learns how much to trust each base model: its predictors are the base models' predicted probabilities, and its coefficients are the weights. We use a plain (unpenalized) logistic regression as the meta-learner.

In [ ]:
model_stacking = StackingClassifier(
                estimators=[('lasso', model_lasso_tuned.best_estimator_),
                            ('rf', model_rf_tuned.best_estimator_),
                            ('knn', model_knn_tuned.best_estimator_)],
                final_estimator=LogisticRegression(C=1e10, max_iter=1000),
                cv=5, stack_method='predict_proba', n_jobs=-1
                )
model_stacking.fit(X_train, y_train)

The coefficients of the meta-learner are the weights of the base models:

In [ ]:
pd.Series(model_stacking.final_estimator_.coef_[0], index=['Lasso', 'Random forest', 'KNN'], name='Weight')

The meta-learner gives KNN almost no weight: it learns to ignore the weak model, which simple averaging cannot do.

In [ ]:
pred_stacked_proba = model_stacking.predict_proba(X_test)[:,1]
roc_auc_score(y_test, pred_stacked_proba)

## Compare models

The test AUCs of the three base models, the simple average and the stacked model side by side:

In [ ]:
pd.DataFrame({'Test AUC': [
                roc_auc_score(y_test, pred_lasso_proba),
                roc_auc_score(y_test, pred_rf_proba),
                roc_auc_score(y_test, pred_knn_proba),
                roc_auc_score(y_test, pred_avg_proba),
                roc_auc_score(y_test, pred_stacked_proba)
            ]}, index=['Lasso', 'Random forest', 'KNN', 'Average of three', 'Stacked']).round(3)

On this split, both ensembles beat every single base model. The stacked model is slightly better than the simple average, because the meta-learner down-weights the weak KNN model instead of giving it a third of the say. An honest caveat: the differences are small, and with only about 250 test applications they are partly noise.

## Your turn!

**Your task:** Add the boosting model from the tree-based models notebook (`HistGradientBoostingClassifier`, tuned with `GridSearchCV`) as a fourth base model. How do the averaged and the stacked test AUC change? Why?

**Bonus:** Use a different meta-learner (e.g., a decision tree or a random forest as `final_estimator`) and compare its test AUC with the logistic regression meta-learner.

In [ ]:
# YOUR CODE HERE